In [1]:
# installing fastcoref (runs on colab's computer)
!pip install -q fastcoref "transformers<5" "datasets<5"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 87.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.8 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# coreference resolution - loading the model
from fastcoref import FCoref

coref_model = FCoref()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/819 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/393 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/362M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/362M [00:00<?, ?B/s]

In [3]:
# coreference resolution - example
text = "Christopher Nolan directed Inception. He also wrote it, and his brother Jonathan helped him with the script."

preds = coref_model.predict(texts=[text])
print(preds[0].get_clusters())

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Inference:   0%|          | 0/1 [00:00<?, ?it/s]

[['Inception', 'it'], ['He', 'his', 'him']]


In [4]:
# coreference resolution - replacing pronouns with names
def resolve_pronouns(text, pred):
    pronouns = {"he", "him", "she", "her", "it", "they", "them"}
    possessives = {"his", "its", "their"}
    changes = []
    for cluster in pred.get_clusters(as_strings=False):
        start, end = cluster[0]
        main = text[start:end]                     # first mention = the real name
        for start, end in cluster[1:]:
            word = text[start:end].lower()
            if word in pronouns:
                changes.append((start, end, main))
            elif word in possessives:
                changes.append((start, end, main + "'s"))
    for start, end, new in sorted(changes, reverse=True):   # replace from the end
        text = text[:start] + new + text[end:]
    return text

print(resolve_pronouns(text, preds[0]))

Christopher Nolan directed Inception. He also wrote Inception, and He's brother Jonathan helped He with the script.


In [7]:
# test review for coreference
review = """I watched The Dark Knight last night. Heath Ledger was incredible as the Joker. He made the character terrifying, and his laugh still gives me chills. Christian Bale was good too, but his voice as Batman sounded strange to me."""

real_pred = coref_model.predict(texts=[review])
for cluster in real_pred[0].get_clusters():
    print(cluster)

print(resolve_pronouns(review, real_pred[0]))

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Inference:   0%|          | 0/1 [00:00<?, ?it/s]

['Heath Ledger', 'He', 'his']
['the Joker', 'the character']
['I', 'me', 'me']
['Christian Bale', 'his']
I watched The Dark Knight last night. Heath Ledger was incredible as the Joker. Heath Ledger made the character terrifying, and Heath Ledger's laugh still gives me chills. Christian Bale was good too, but Christian Bale's voice as Batman sounded strange to me.
